# Good Code

## コード品質の柱

『Good Code，Bad Code』では次の6つをコードの品質を支える柱としている

:::{admonition} コード品質の柱

1. コードを読みやすくする
2. 想定外の事態をなくす
3. 誤用しにくいコードを書く
4. コードをモジュール化する
5. コードを再利用、汎用化しやすくする
6. テストしやすいコードを書き、適切にテストする

:::


## 抽象化レイヤー

コードを上手く構造化できれば、きれいな抽象化レイヤーができる。


### 良い抽象化の例

例えばHTTPでPOSTリクエストを送るとする。`requests`パッケージを使えば次のように書くことができる。

```python
import requests
requests.post('https://httpbin.org/post', data={'key': 'value'})
```

これは内部的には

- データのシリアライズ
- HTTPプロトコルに関する様々な問題の解決
- TCP通信
- データ送信時のエラーと訂正

など様々なことを行っているが、適切に抽象化されているためユーザーは内部実装を気にせず、やりたいことだけを考えて短い記述だけすればよくなっている。

きれいな抽象化レイヤーを作ることで、読みやすく、モジュール性があり、再利用性があり、テスタビリティの高いコードになる。


### コードのレイヤー

抽象化レイヤーを作るにはコードを関数やクラス、パッケージなどの異なる単位に分割することになる。



#### 関数

例えば「車の所有者の住所を探し、見つかれば手紙を送る」という処理を考える

##### 例：悪いコード

```py
from typing import Optional


def send_owner_a_letter(
    vehicle: Vehicle,
    letter: Letter,
) -> Optional[Confirmation]:
    owners_address: Optional[Address] = None

    # 住所を探す基本的なロジック
    if vehicle.has_been_scrapped():
        owners_address = SCRAPYARD_ADDRESS
    else:
        most_recent_purchase = vehicle.get_most_recent_purchase()

        if most_recent_purchase is None:
            owners_address = SHOWROOM_ADDRESS
        else:
            owners_address = most_recent_purchase.get_buyers_address()

    # 条件付きで手紙を送るロジック
    if owners_address is None:
        return None

    return send_letter(owners_address, letter)
```

このコードでは1つの関数に「住所を探す」と「手紙を送るかどうかの判定」が混在しており、関数名と一致していない

##### 例：良いコード

関数を小さくして分割し、関数名が内容と一致するようにすることでより読みやすくなる

```py
from typing import Optional


def find_owners_address(vehicle: Vehicle) -> Optional[Address]:
    if vehicle.has_been_scrapped():
        return SCRAPYARD_ADDRESS

    most_recent_purchase = vehicle.get_most_recent_purchase()

    if most_recent_purchase is None:
        return SHOWROOM_ADDRESS

    return most_recent_purchase.get_buyers_address()


def send_owner_a_letter(
    vehicle: Vehicle,
    letter: Letter,
) -> Optional[Confirmation]:
    owners_address = find_owners_address(vehicle)

    if owners_address is None:
        return None

    return send_letter(owners_address, letter)
```


:::{admonition} 関数作成のポイント

- 単一のタスクをおこなう
- 適切な名前をつける

:::

#### クラス


:::{admonition} クラス作成のポイント

- 凝集：クラス内のオブジェクトが適切なクラスに所属している度合い
- 関心の分離
- 行数：例えば300行を超えるクラスは多くの場合、多くの概念を含みすぎており凝集度が低いとされる

:::

## 契約プログラミング

契約プログラミング、または 契約による設計（Design by Contract: DbC）とは、関数やクラスの利用者と実装者の間に、明確な「契約」を定めてソフトウェアを設計する考え方。

### 契約の3要素

契約は主に次の3要素で構成される：

#### 1. 事前条件

処理を呼び出す側が満たすべき条件。  
例えば「引き出し金額は0より大きい」「配列は空でない」など。

```py
def withdraw(balance: int, amount: int) -> int:
    assert amount > 0, "引き出し金額は正数でなければならない"
    assert amount <= balance, "残高を超えて引き出すことはできない"

    return balance - amount
```

#### 2. 事後条件

処理が正常に終了したとき、実装側が保証する条件。

```py
def withdraw(balance: int, amount: int) -> int:
    assert amount > 0
    assert amount <= balance

    new_balance = balance - amount

    assert new_balance == balance - amount
    assert new_balance >= 0

    return new_balance
```

この関数は、呼び出し後に以下を保証する。

- 新しい残高は元の残高から引き出し額を引いた値である（`new_balance == balance - amount`）
- 残高は負にならない（`new_balance >= 0`）


#### 3. 不変条件

コードの呼び出し前後で変わらず常に成立していなければならない条件。

例えば次のクラスがあるとする：

```py
class BankAccount:
    def __init__(self, balance: int = 0) -> None:
        assert balance >= 0
        self._balance = balance
        self._check_invariant()

    def withdraw(self, amount: int) -> None:
        assert amount > 0
        assert amount <= self._balance

        self._balance -= amount

        self._check_invariant()

    def deposit(self, amount: int) -> None:
        assert amount > 0

        self._balance += amount

        self._check_invariant()

    def _check_invariant(self) -> None:
        assert self._balance >= 0, "残高は常に0以上でなければならない"
```

このクラスでは、 `self._balance >= 0` が不変条件


| 契約   | 保証する側   | 違反した場合            |
| ---- | ------- | ----------------- |
| 事前条件 | 呼び出し側   | 呼び出し方が誤っている       |
| 事後条件 | 実装側     | 関数の実装が誤っている       |
| 不変条件 | クラスの実装側 | オブジェクトの状態管理が誤っている |


ただし、外部入力に対して assert を使うのは一般に不適切。Pythonの assert は、最適化オプションを付けて実行すると無効化される可能性があるため。

したがって、実務では次のように使い分ける。

- ユーザー入力やAPI入力の検証：ValueError やバリデーションライブラリ
- 内部実装上、必ず成立すべき前提：assert
- ドメイン上のルール違反：専用の例外

## 参考

- Good Code， Bad Code ～持続可能な開発のためのソフトウェアエンジニア的思考. (2023). 株式会社秀和システム.